In [ ]:
import googlemaps
import os
from datetime import datetime
import pandas as pd
from dotenv import load_dotenv

# 1. Configuración Inicial
load_dotenv()
api_key = os.getenv('GOOGLE_MAPS_API_KEY')
gmaps = googlemaps.Client(key=api_key)

# Origen: Francia y Pellegrini
origen_query = "Terminal de Ómnibus Mariano Moreno, Rosario"

# Radio: Francia y Pellegrini
radio_query = "MO Pet Shopping - Sucursal Echesortu"
res_radio = gmaps.geocode(radio_query)
location = res_radio[0]['geometry']['location']

# 2. Definición de Categorías Refinadas
# Añadimos keywords específicas para "comida de mesa"
categorias_busqueda = [
    {
        'label': 'Restaurante', 
        'type': 'restaurant', 
        'keyword': 'parrilla cena almuerzo gourmet pizza' # Keywords para filtrar comida al paso
    },
    {
        'label': 'Museo', 
        'type': 'museum', 
        'keyword': 'museo'
    },
    {
        'label': 'Heladería', 
        'type': None, 
        'keyword': 'heladeria artesanal'
    },
    {
        'label': 'Cervecería', 
        'type': 'bar', 
        'keyword': 'cerveceria artesanal comida'
    }
]

datos_lugares = []
place_ids_vistos = set()

print(f"Buscando lugares para comer y visitar en Rosario...")

# 3. Bucle de Búsqueda
for cat in categorias_busqueda:
    busqueda = gmaps.places_nearby(
        location=location,
        radius=3000,
        type=cat['type'],
        keyword=cat['keyword'],
        language='es'
    )

    for item in busqueda.get('results', []):
        place_id = item['place_id']
        
        if place_id in place_ids_vistos:
            continue
        
        # --- FILTRO CRÍTICO ---
        # Si es un restaurante, descartamos los que Google etiqueta como 
        # 'meal_takeaway' o 'bakery' si NO tienen también la etiqueta 'restaurant'
        tipos_del_lugar = item.get('types', [])
        if cat['label'] == 'Restaurante':
            if 'meal_takeaway' in tipos_del_lugar and 'restaurant' not in tipos_del_lugar:
                continue # Salta este lugar si es solo para llevar
        # ----------------------

        place_ids_vistos.add(place_id)

        try:
            detalles_res = gmaps.place(
                place_id=place_id, 
                fields=['name', 'geometry', 'formatted_address', 'rating', 'opening_hours'],
                language='es'
            ).get('result', {})

            # Procesamiento de Horarios (Igual al anterior)
            dias_semana = ['Lunes', 'Martes', 'Miércoles', 'Jueves', 'Viernes', 'Sábado', 'Domingo']
            horarios_semanales = {dia: [] for dia in dias_semana}
            
            periodos = detalles_res.get('opening_hours', {}).get('periods', [])

            if periodos:
                for p in periodos:
                    idx_google = p['open']['day']
                    idx_nuestro = (idx_google - 1) % 7 
                    nombre_dia = dias_semana[idx_nuestro]
                    h_abre = p['open']['time']
                    h_cierra = p.get('close', {}).get('time', '24h')
                    horarios_semanales[nombre_dia].append({
                        'apertura': f"{h_abre[:2]}:{h_abre[2:]}",
                        'cierre': f"{h_cierra[:2]}:{h_cierra[2:]}" if h_cierra != '24h' else '24h'
                    })
                for dia in dias_semana:
                    if not horarios_semanales[dia]: horarios_semanales[dia] = "Cerrado"
            else:
                horarios_semanales = "No disponible"

            # 4. Guardar en la lista final
            datos_lugares.append({
                'nombre': detalles_res.get('name'),
                'tipo': cat['label'],
                'coords': detalles_res.get('geometry', {}).get('location'),
                'direccion': detalles_res.get('formatted_address'),
                'puntaje': detalles_res.get('rating', 0),
                'horarios_semana': horarios_semanales
            })
        except Exception as e:
            continue

# 5. Ordenar por Puntaje para mostrar lo mejor primero
df = pd.DataFrame(datos_lugares)
df = df.sort_values(by='puntaje', ascending=False)

print(f"\nSe encontraron {len(df)} lugares de alta relevancia.")
print(df[['nombre', 'tipo', 'puntaje']])

Buscando lugares para comer y visitar en Rosario...

Se encontraron 75 lugares de alta relevancia.
                                               nombre         tipo  puntaje
34    Museo y archivo ferroviario regional de Rosario        Museo      5.0
28                        Deposito Museo de la Ciudad        Museo      5.0
22  MIAPCR MUSEUM Musée International d'Art Post-C...        Museo      5.0
45                                     Rumo Heladería    Heladería      4.9
27                         Museo de Arte Sacro Barnes        Museo      4.8
..                                                ...          ...      ...
56                        Brooklyn | Crafters' Garden   Cervecería      4.0
72                           Peñón del Águila Rosario   Cervecería      4.0
19                       Restaurante Bendita Costilla  Restaurante      3.9
29                                  Museo de la Salud        Museo      3.7
71                                          Ipa House   Cervecerí

In [11]:
print(f"\nSe encontraron {len(df)} lugares de alta relevancia.")
print(df[['nombre', 'tipo', 'puntaje', 'coords']])


Se encontraron 27 lugares de alta relevancia.
                                               nombre         tipo  puntaje  \
2   MIAPCR MUSEUM Musée International d'Art Post-C...        Museo      5.0   
10                                             AiLilú    Heladería      5.0   
7                                       "WOW! Gelato"    Heladería      4.8   
26                               Ámsterdam Cervecería   Cervecería      4.8   
3                                 Heladería "Catania"    Heladería      4.7   
16                     El Paraíso de la Birra Rosario   Cervecería      4.7   
21                           Anfora Cerveza Artesanal   Cervecería      4.7   
17                               Mosto Somos Cervezas   Cervecería      4.6   
24                                    Mosto Echesortu   Cervecería      4.6   
18                                 ANIMAL BEER MARKET   Cervecería      4.5   
4                             RIO Helados - Echesortu    Heladería      4.5   
23   

In [4]:
#guardar la matriz_res
#df = pd.DataFrame(matriz_res['rows'])
#df.to_csv('matriz_res.csv', index=False)

import pickle
with open('datos_lugares.pkl', 'wb') as f:
    pickle.dump(datos_lugares, f)

In [5]:
with open('datos_lugares.pkl', 'rb') as f:
    datos_lugares = pickle.load(f)

In [11]:
# Cálculo de la Matriz "Todos contra Todos"
coords_solo = [d['coords'] for d in datos_lugares]
nombres_solo = [d['nombre'] for d in datos_lugares]

matriz_res = gmaps.distance_matrix(
    origins=coords_solo,
    destinations=coords_solo,
    mode="driving", 
    departure_time=datetime.now()
)

# Estructurar la Matriz de Distancias en un DataFrame
distancias_data = []
for fila in matriz_res['rows']:
    distancias_data.append([elemento['distance']['value'] for elemento in fila['elements']])

df_distancias = pd.DataFrame(distancias_data, index=nombres_solo, columns=nombres_solo)

# Estructurar la Matriz de tiempos en un DataFrame
tiempos_data = []
for fila in matriz_res['rows']:
    tiempos_data.append([elemento['duration']['value'] for elemento in fila['elements']])

df_tiempos = pd.DataFrame(tiempos_data, index=nombres_solo, columns=nombres_solo)


# Mostrar Información de Horarios y la Matriz
print("--- HORARIOS DE APERTURA ---")
for lugar in datos_lugares:
    print(f"{lugar['nombre']}:")
    # Imprimimos solo el horario de hoy para no saturar la consola
    print(f"  {lugar['horarios']}\n")

--- HORARIOS DE APERTURA ---
Rosario:
  ['Monday: Open 24 hours', 'Tuesday: Open 24 hours', 'Wednesday: Open 24 hours', 'Thursday: Open 24 hours', 'Friday: Open 24 hours', 'Saturday: Open 24 hours', 'Sunday: Open 24 hours']

El Cairo:
  ['Monday: 8:00\u202fAM\u2009–\u200911:00\u202fPM', 'Tuesday: 8:00\u202fAM\u2009–\u200911:00\u202fPM', 'Wednesday: 8:00\u202fAM\u2009–\u200911:00\u202fPM', 'Thursday: 8:00\u202fAM\u2009–\u200911:00\u202fPM', 'Friday: 8:00\u202fAM\u2009–\u200911:00\u202fPM', 'Saturday: 8:00\u202fAM\u2009–\u200911:00\u202fPM', 'Sunday: 4:00\u2009–\u200911:00\u202fPM']

Alto Rosario Shopping:
  ['Monday: 10:00\u202fAM\u2009–\u200912:00\u202fAM', 'Tuesday: 10:00\u202fAM\u2009–\u200912:00\u202fAM', 'Wednesday: 10:00\u202fAM\u2009–\u200912:00\u202fAM', 'Thursday: 10:00\u202fAM\u2009–\u200912:00\u202fAM', 'Friday: 10:00\u202fAM\u2009–\u20091:00\u202fAM', 'Saturday: 10:00\u202fAM\u2009–\u20091:00\u202fAM', 'Sunday: 10:00\u202fAM\u2009–\u200912:00\u202fAM']

Museo Municipal de Be

In [6]:
matriz_res['rows'][0]['elements'][2]

{'distance': {'text': '3.0 km', 'value': 2952},
 'duration': {'text': '10 mins', 'value': 613},
 'duration_in_traffic': {'text': '9 mins', 'value': 530},
 'status': 'OK'}

In [12]:
df_distancias

,Rosario,El Cairo,Alto Rosario Shopping,Museo Municipal de Bellas Artes Juan B. Castagnino,VIP Rosario,Ruffo Coffee Co.
Rosario,0,4111,2952,3452,5028,2394
El Cairo,3153,0,5264,2880,1924,2696
Alto Rosario Shopping,2360,4632,0,4871,5217,2591
Museo Municipal de Bellas Artes Juan B. Castagnino,3038,3399,4633,0,4576,2454
VIP Rosario,4926,1692,6390,3479,0,4746
Ruffo Coffee Co.,2001,2767,2046,2651,3862,0


In [13]:
df_tiempos

,Rosario,El Cairo,Alto Rosario Shopping,Museo Municipal de Bellas Artes Juan B. Castagnino,VIP Rosario,Ruffo Coffee Co.
Rosario,0,787,613,613,918,412
El Cairo,554,0,894,585,395,506
Alto Rosario Shopping,520,847,0,912,864,485
Museo Municipal de Bellas Artes Juan B. Castagnino,503,741,857,0,868,512
VIP Rosario,935,356,996,630,0,806
Ruffo Coffee Co.,372,560,433,551,646,0


In [ ]:
# Guardar df_distancias en un archivo CSV
df_distancias.to_csv('distancias.csv')

# Guardar df_tiempos en un archivo CSV
df_tiempos.to_csv('tiempos.csv')
